# Phase 1: Exploratory Data Analysis (EDA)

**Goal**: Understand the IEEE-CIS Fraud Detection dataset before building models.

We investigate:
1. Dataset shape, fraud rate, time span
2. Missing values patterns
3. Transaction amount distributions (fraud vs legit)
4. Pseudo-user key behavior
5. Time-based split verification
6. Feature correlations

In [ ]:
import sys
sys.path.insert(0, '..')  # so we can import src

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import load_config
from src.training.data_loader import (
    load_raw_data, merge_data, build_pseudo_user_key,
    compute_transaction_datetime, time_based_split, prepare_dataset
)

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)

cfg = load_config()

## 1. Load Data

In [ ]:
txn, identity = load_raw_data(cfg)
df = merge_data(txn, identity)

print(f"Transaction table: {txn.shape}")
print(f"Identity table: {identity.shape}")
print(f"Merged: {df.shape}")
print(f"\nIdentity coverage: {len(identity) / len(txn):.1%} of transactions")

## 2. Fraud Rate

In [ ]:
fraud_rate = df['isFraud'].mean()
n_fraud = df['isFraud'].sum()
n_legit = len(df) - n_fraud

print(f"Fraud rate: {fraud_rate:.4f} ({fraud_rate:.2%})")
print(f"Fraud: {n_fraud:,} | Legit: {n_legit:,} | Total: {len(df):,}")
print(f"Class ratio (legit:fraud): {n_legit/n_fraud:.1f}:1")

fig, ax = plt.subplots(figsize=(6, 4))
df['isFraud'].value_counts().plot(kind='bar', ax=ax, color=['steelblue', 'crimson'])
ax.set_xticklabels(['Legit (0)', 'Fraud (1)'], rotation=0)
ax.set_ylabel('Count')
ax.set_title(f'Class Distribution (Fraud Rate: {fraud_rate:.2%})')
for i, v in enumerate(df['isFraud'].value_counts().values):
    ax.text(i, v + 1000, f'{v:,}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Missing Values

In [ ]:
missing = df.isnull().mean().sort_values(ascending=False)
missing_high = missing[missing > 0.5]
missing_moderate = missing[(missing > 0.1) & (missing <= 0.5)]
missing_low = missing[(missing > 0) & (missing <= 0.1)]

print(f"Columns with >50% missing: {len(missing_high)}")
print(f"Columns with 10-50% missing: {len(missing_moderate)}")
print(f"Columns with <10% missing: {len(missing_low)}")
print(f"Columns with 0% missing: {(missing == 0).sum()}")

# Plot top 30 missing
fig, ax = plt.subplots(figsize=(12, 8))
missing.head(30).plot(kind='barh', ax=ax, color='coral')
ax.set_xlabel('Missing Fraction')
ax.set_title('Top 30 Columns by Missing Rate')
ax.axvline(x=0.5, color='red', linestyle='--', alpha=0.7, label='50% threshold')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Time Span and Transaction Volume

In [ ]:
df = compute_transaction_datetime(df)

print(f"Time range: {df['transaction_datetime'].min()} to {df['transaction_datetime'].max()}")
print(f"Span: {(df['TransactionDT'].max() - df['TransactionDT'].min()) / 86400:.0f} days")

# Daily transaction volume
daily = df.groupby('transaction_date').agg(
    total=('isFraud', 'count'),
    fraud=('isFraud', 'sum')
).reset_index()
daily['fraud_rate'] = daily['fraud'] / daily['total']

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].bar(daily['transaction_date'], daily['total'], color='steelblue', alpha=0.7)
axes[0].set_ylabel('Transactions / Day')
axes[0].set_title('Daily Transaction Volume')

axes[1].plot(daily['transaction_date'], daily['fraud_rate'], color='crimson', linewidth=1.5)
axes[1].axhline(y=fraud_rate, color='gray', linestyle='--', alpha=0.5, label=f'Overall: {fraud_rate:.3f}')
axes[1].set_ylabel('Fraud Rate')
axes[1].set_title('Daily Fraud Rate')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Transaction Amount Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Log-scale amounts by fraud label
for label, color, name in [(0, 'steelblue', 'Legit'), (1, 'crimson', 'Fraud')]:
    subset = df[df['isFraud'] == label]['TransactionAmt']
    axes[0].hist(np.log1p(subset), bins=100, alpha=0.6, color=color, label=name, density=True)

axes[0].set_xlabel('log(1 + TransactionAmt)')
axes[0].set_ylabel('Density')
axes[0].set_title('Transaction Amount Distribution (log scale)')
axes[0].legend()

# Box plot
df.boxplot(column='TransactionAmt', by='isFraud', ax=axes[1],
           showfliers=False)
axes[1].set_xlabel('isFraud')
axes[1].set_ylabel('TransactionAmt')
axes[1].set_title('Amount by Fraud Label (outliers hidden)')
plt.suptitle('')  # remove auto-title from boxplot

plt.tight_layout()
plt.show()

# Summary stats
print(df.groupby('isFraud')['TransactionAmt'].describe())

## 6. Pseudo-User Key Analysis

We built a pseudo-user key from `card1 + addr1 + account_start_day`.
Let's check if it behaves like a real user ID — most users should have
a small number of transactions, with a few power users.

In [ ]:
df = build_pseudo_user_key(df)

user_txn_counts = df.groupby('pseudo_user_id').size()
user_fraud_rates = df.groupby('pseudo_user_id')['isFraud'].mean()

print(f"Unique pseudo-users: {len(user_txn_counts):,}")
print(f"\nTransactions per user:")
print(user_txn_counts.describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Transactions per user
axes[0].hist(user_txn_counts.clip(upper=50), bins=50, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Transactions per Pseudo-User (capped at 50)')
axes[0].set_ylabel('Count')
axes[0].set_title('Pseudo-User Transaction Distribution')

# Fraud rate per user (only users with enough data)
axes[1].hist(user_fraud_rates, bins=50, color='crimson', edgecolor='white')
axes[1].set_xlabel('Fraud Rate per Pseudo-User')
axes[1].set_ylabel('Count')
axes[1].set_title('Fraud Rate Distribution Across Pseudo-Users')

plt.tight_layout()
plt.show()

# Check: do some users have ALL fraud or ALL legit?
all_fraud = (user_fraud_rates == 1.0).sum()
all_legit = (user_fraud_rates == 0.0).sum()
mixed = len(user_fraud_rates) - all_fraud - all_legit
print(f"\nUsers with 100% fraud: {all_fraud:,}")
print(f"Users with 0% fraud: {all_legit:,}")
print(f"Users with mixed: {mixed:,}")

## 7. Time-Based Split Verification

In [ ]:
train, val, test = time_based_split(df, cfg.train_ratio, cfg.val_ratio)

for name, split in [('Train', train), ('Val', val), ('Test', test)]:
    print(f"{name}: {len(split):>7,} rows | Fraud rate: {split['isFraud'].mean():.4f} | "
          f"Time: {split['transaction_datetime'].min().date()} → {split['transaction_datetime'].max().date()}")

# Visualize the split
fig, ax = plt.subplots(figsize=(14, 4))
for name, split, color in [('Train', train, 'steelblue'), ('Val', val, 'orange'), ('Test', test, 'green')]:
    daily = split.groupby('transaction_date').size()
    ax.bar(daily.index, daily.values, color=color, alpha=0.7, label=name)

ax.set_xlabel('Date')
ax.set_ylabel('Transactions')
ax.set_title('Time-Based Split: Train / Validation / Test')
ax.legend()
plt.tight_layout()
plt.show()

# WHY time-based split?
print("\n--- Why time-based split (not random)? ---")
print("Random split would let the model see future fraud patterns during training.")
print("In production, we only have past data to predict future transactions.")
print("Time-split simulates this: train on past, evaluate on future.")

## 8. Hourly and Day-of-Week Patterns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Hourly fraud rate
hourly = df.groupby('transaction_hour')['isFraud'].mean()
axes[0].bar(hourly.index, hourly.values, color='crimson', alpha=0.7)
axes[0].axhline(y=fraud_rate, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Fraud Rate')
axes[0].set_title('Fraud Rate by Hour')

# Day of week fraud rate
dow = df.groupby('transaction_dayofweek')['isFraud'].mean()
axes[1].bar(dow.index, dow.values, color='crimson', alpha=0.7)
axes[1].axhline(y=fraud_rate, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Day of Week (0=Mon)')
axes[1].set_ylabel('Fraud Rate')
axes[1].set_title('Fraud Rate by Day of Week')

plt.tight_layout()
plt.show()

## 9. Product Code and Card Brand Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Product code
if 'ProductCD' in df.columns:
    prod_fraud = df.groupby('ProductCD')['isFraud'].agg(['mean', 'count']).sort_values('mean', ascending=False)
    axes[0].bar(prod_fraud.index, prod_fraud['mean'], color='crimson', alpha=0.7)
    axes[0].axhline(y=fraud_rate, color='gray', linestyle='--')
    axes[0].set_title('Fraud Rate by Product Code')
    axes[0].set_ylabel('Fraud Rate')

# Card brand
if 'card4' in df.columns:
    card_fraud = df.groupby('card4')['isFraud'].agg(['mean', 'count']).sort_values('mean', ascending=False)
    axes[1].bar(card_fraud.index, card_fraud['mean'], color='steelblue', alpha=0.7)
    axes[1].axhline(y=fraud_rate, color='gray', linestyle='--')
    axes[1].set_title('Fraud Rate by Card Brand')
    axes[1].set_ylabel('Fraud Rate')

plt.tight_layout()
plt.show()

## 10. Summary

Key findings from EDA:
- **Fraud rate**: ~3.5% — imbalanced but not extremely rare. Use PR-AUC, class weights.
- **Time span**: ~6 months. Time-based split is essential.
- **Missing values**: Many V-columns have >50% missing. XGBoost handles NaN natively.
- **Pseudo-user key**: Produces a plausible user distribution (most users have few txns).
- **Fraud patterns**: Vary by hour, day of week, product code, and card brand.
- **Amount**: Fraud transactions tend to have different amount distributions.